In [1]:
"""Organise and filter relevant data from datasets to collect important info."""

# Import libraries
from pathlib import Path

import pandas as pd

In [2]:
# Set relevant datasets
TEST_2020 = 'datasets/air_quality/2020s/2020_01.csv'
TEST_2019 = 'datasets/air_quality/pre_2020/2019_01.csv'
# Code number used for NO2
POLLUTANT_CODE = 8
# Station data file
STATION_FILE = 'datasets/stations/2022.csv'

In [3]:
def load_csv_file_as_data_f(path: str) -> pd.DataFrame:
    """Load CSV file.

    Args:
        path (str): Path to file

    Returns:
        pd.DataFrame: Pandas Dataframe.

    """
    return pd.read_csv(path, sep=',')

In [4]:
original_csv = load_csv_file_as_data_f(TEST_2020)
original_csv


,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
0,8,Barcelona,19,Barcelona,4,7,2020,1,1,3.0,...,18.0,V,40.0,V,44.0,V,31.0,V,33.0,V
1,8,Barcelona,19,Barcelona,4,7,2020,1,2,13.0,...,18.0,V,10.0,V,3.0,V,57.0,V,NaN,N
2,8,Barcelona,19,Barcelona,4,7,2020,1,3,33.0,...,56.0,V,46.0,V,40.0,V,32.0,V,NaN,N
3,8,Barcelona,19,Barcelona,4,7,2020,1,4,12.0,...,5.0,V,7.0,V,2.0,V,3.0,V,NaN,N
4,8,Barcelona,19,Barcelona,4,7,2020,1,5,1.0,...,73.0,V,53.0,V,33.0,V,29.0,V,25.0,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1274,8,Barcelona,19,Barcelona,58,14,2020,1,27,60.0,...,60.0,V,58.0,V,60.0,V,61.0,V,NaN,N
1275,8,Barcelona,19,Barcelona,58,14,2020,1,28,58.0,...,62.0,V,58.0,V,61.0,V,65.0,V,NaN,N
1276,8,Barcelona,19,Barcelona,58,14,2020,1,29,70.0,...,24.0,V,42.0,V,49.0,V,50.0,V,51.0,V
1277,8,Barcelona,19,Barcelona,58,14,2020,1,30,36.0,...,46.0,V,56.0,V,61.0,V,60.0,V,56.0,V


In [5]:
def filter_pollutant(orig_df: pd.DataFrame) -> pd.DataFrame:
    """Filter to only NO2 relevant stats.

    Args:
        orig_df (pd.DataFrame): The original CSV dataframe.

    Returns:
        pd.DataFrame: The data filtered to NO2.

    """
    return orig_df[orig_df['CODI_CONTAMINANT'] == POLLUTANT_CODE].copy()

In [6]:
filtered_csv = filter_pollutant(original_csv)
filtered_csv

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,64.0,V,66.0,V,63.0,V,56.0,V,48.0,V
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,67.0,V,61.0,V,49.0,V,60.0,V,NaN,N
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,43.0,V,40.0,V,39.0,V,38.0,V,NaN,N
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,37.0,V,36.0,V,22.0,V,20.0,V,NaN,N
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,73.0,V,69.0,V,61.0,V,54.0,V,49.0,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,8.0,V,11.0,V,9.0,V,8.0,V,NaN,N
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,10.0,V,11.0,V,7.0,V,4.0,V,NaN,N
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,46.0,V,27.0,V,21.0,V,18.0,V,15.0,V
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,15.0,V,7.0,V,5.0,V,5.0,V,8.0,V


In [7]:
def calc_pollute_daily_avg(filtered_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate average NO2 per day.

    Args:
        filtered_df (pd.DataFrame): Data frame with only relevant stations.

    Returns:
        pd.DataFrame: Dataframe with average NO2 values.

    """
    # Get the hour columns with values for NO2.
    hour_cols = [col for col in filtered_df.columns if col.startswith('H')]
    # Get the V cols which define whethere a value was gathered.
    valid_cols = [col for col in filtered_df.columns if col.startswith('V')]
    # Choose to drop any invalid values listed as "N" in the V column.
    for hour, value in zip(hour_cols, valid_cols, strict=False):
        filtered_df.loc[filtered_df[value] == 'N', hour] = None

    # Compute mean of valid hourly values
    filtered_df['no2_daily_avg'] = filtered_df[hour_cols].mean(axis=1)

    return filtered_df

In [8]:
calc_pollute_daily_avg(filtered_csv)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,V20,H21,V21,H22,V22,H23,V23,H24,V24,no2_daily_avg
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,V,66.0,V,63.0,V,56.0,V,48.0,V,32.291667
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,V,61.0,V,49.0,V,60.0,V,NaN,N,38.565217
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,V,40.0,V,39.0,V,38.0,V,NaN,N,38.956522
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,V,36.0,V,22.0,V,20.0,V,NaN,N,33.217391
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,V,69.0,V,61.0,V,54.0,V,49.0,V,29.291667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,V,11.0,V,9.0,V,8.0,V,NaN,N,8.000000
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,V,11.0,V,7.0,V,4.0,V,NaN,N,6.782609
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,V,27.0,V,21.0,V,18.0,V,15.0,V,19.130435
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,V,7.0,V,5.0,V,5.0,V,8.0,V,9.333333


In [9]:
def create_date_column(filtered_df: pd.DataFrame) -> pd.DataFrame:
    """Generate a date column from given values.

    Args:
        filtered_df (pd.DataFrame): Dataframe with filtered stations.

    Returns:
        pd.DataFrame: Dataframe with a clear date column.

    """
    filtered_df['date'] = pd.to_datetime(
        {
            'year': filtered_df['ANY'],
            'month': filtered_df['MES'],
            'day': filtered_df['DIA'],
        }
    )
    return filtered_df


In [10]:
create_date_column(filtered_csv)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H21,V21,H22,V22,H23,V23,H24,V24,no2_daily_avg,date
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,66.0,V,63.0,V,56.0,V,48.0,V,32.291667,2020-01-01
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,61.0,V,49.0,V,60.0,V,NaN,N,38.565217,2020-01-02
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,40.0,V,39.0,V,38.0,V,NaN,N,38.956522,2020-01-03
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,36.0,V,22.0,V,20.0,V,NaN,N,33.217391,2020-01-04
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,69.0,V,61.0,V,54.0,V,49.0,V,29.291667,2020-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,11.0,V,9.0,V,8.0,V,NaN,N,8.000000,2020-01-27
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,11.0,V,7.0,V,4.0,V,NaN,N,6.782609,2020-01-28
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,27.0,V,21.0,V,18.0,V,15.0,V,19.130435,2020-01-29
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,7.0,V,5.0,V,5.0,V,8.0,V,9.333333,2020-01-30


In [11]:
def load_station_file(file_path: str) -> pd.DataFrame:
    """Load a station data file and builds its location.

    Args:
        file_path (string): path to the file

    Returns:
        pd.DataFrame: the list of station numbers and locations.

    """
    data_f = pd.read_csv(file_path, sep=',')

    # Filter for the pollutant
    data_f_no2 = data_f[data_f['Codi_Contaminant'] == POLLUTANT_CODE].copy()

    # Rename the columns
    data_f_no2 = data_f_no2.rename(
        columns={
            'Estacio': 'estacio',
            'nom_cabina': 'station_name',
            'Longitud': 'lon',
            'Latitud': 'lat',
        }
    )

    return data_f_no2[['estacio', 'station_name', 'lat', 'lon']]


In [12]:
station_info = load_station_file(STATION_FILE)
station_info

,estacio,station_name,lat,lon
0,50,Barcelona - Ciutadella,41.38640,2.1874
4,43,Barcelona - Eixample,41.38530,2.1538
11,44,Barcelona - Gràcia,41.39870,2.1534
18,57,Barcelona - Palau Reial,41.38750,2.1151
25,4,Barcelona - Poblenou,41.40390,2.2045
29,42,Barcelona - Sants,41.37880,2.1331
32,54,Barcelona - Vall Hebron,41.42610,2.1480
39,58,Barcelona - Observatori Fabra,41.41843,2.1239


In [13]:
def merge_station_data(
    data_f_pollute: pd.DataFrame, station_data: pd.DataFrame
) -> pd.DataFrame:
    """Merge station data with pollution data.

    Args:
        data_f_pollute (pd.DataFrame): Dataframe with pollution data.
        station_data (pd.DataFrame): Dataframe with station data.

    Returns:
        pd.DataFrame: Merged dataframe.

    """
    data_f_pollute = data_f_pollute.rename(columns={'ESTACIO': 'estacio'})

    return data_f_pollute.merge(station_data, on='estacio', how='left')


In [14]:
merge_station_data(filtered_csv, station_info)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,estacio,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,V22,H23,V23,H24,V24,no2_daily_avg,date,station_name,lat,lon
0,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,V,56.0,V,48.0,V,32.291667,2020-01-01,Barcelona - Poblenou,41.40390,2.2045
1,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,V,60.0,V,NaN,N,38.565217,2020-01-02,Barcelona - Poblenou,41.40390,2.2045
2,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,V,38.0,V,NaN,N,38.956522,2020-01-03,Barcelona - Poblenou,41.40390,2.2045
3,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,V,20.0,V,NaN,N,33.217391,2020-01-04,Barcelona - Poblenou,41.40390,2.2045
4,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,V,54.0,V,49.0,V,29.291667,2020-01-05,Barcelona - Poblenou,41.40390,2.2045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,V,8.0,V,NaN,N,8.000000,2020-01-27,Barcelona - Observatori Fabra,41.41843,2.1239
228,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,V,4.0,V,NaN,N,6.782609,2020-01-28,Barcelona - Observatori Fabra,41.41843,2.1239
229,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,V,18.0,V,15.0,V,19.130435,2020-01-29,Barcelona - Observatori Fabra,41.41843,2.1239
230,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,V,5.0,V,8.0,V,9.333333,2020-01-30,Barcelona - Observatori Fabra,41.41843,2.1239


In [15]:
def process_pollution_file(
    file_path: str, station_file: str = STATION_FILE
) -> pd.DataFrame:
    """Run the full NO2 processing pipeline for a single CSV file.

    Args:
        file_path (Path | str): Path to the air quality CSV file.
        station_file (str): Path to the station metadata file.

    Returns:
        pd.DataFrame: Merged dataframe with NO2 daily averages and station info.

    """
    orig_df = load_csv_file_as_data_f(file_path)
    filtered_df = filter_pollutant(orig_df)
    filtered_df = calc_pollute_daily_avg(filtered_df)
    filtered_df = create_date_column(filtered_df)
    station_df = load_station_file(station_file)

    return merge_station_data(filtered_df, station_df)


In [16]:
processed_df = process_pollution_file(TEST_2020, STATION_FILE)

In [17]:
def clean_final_columns(final_data_f: pd.DataFrame) -> pd.DataFrame:
    """Clean columns to essential data.

    Args:
        final_data_f (pd.DataFrame): Data frames with all info added.

    Returns:
        pd.DataFrame: Cleaned data frame with only essential info.

    """
    final_data_f = final_data_f.rename(
        columns={'ANY': 'year', 'MES': 'month', 'DIA': 'day'}
    )

    # Step 2: Decide which columns to keep
    keep_cols = [
        'date',
        'year',
        'month',
        'day',
        'estacio',
        'station_name',
        'lat',
        'lon',
        'no2_daily_avg',
    ]

    # Keep only columns that exist.
    keep_cols = [col for col in keep_cols if col in final_data_f.columns]

    # Step 3: Return cleaned DataFrame
    return final_data_f[keep_cols].sort_values(['date']).reset_index(drop=True)


In [18]:
cleaned_df = clean_final_columns(processed_df)
cleaned_df

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2020-01-01,2020,1,1,4,Barcelona - Poblenou,41.40390,2.2045,32.291667
1,2020-01-01,2020,1,1,43,Barcelona - Eixample,41.38530,2.1538,36.833333
2,2020-01-01,2020,1,1,42,Barcelona - Sants,41.37880,2.1331,25.000000
3,2020-01-01,2020,1,1,54,Barcelona - Vall Hebron,41.42610,2.1480,36.958333
4,2020-01-01,2020,1,1,58,Barcelona - Observatori Fabra,41.41843,2.1239,14.333333
...,...,...,...,...,...,...,...,...,...
227,2020-01-31,2020,1,31,43,Barcelona - Eixample,41.38530,2.1538,79.384615
228,2020-01-31,2020,1,31,42,Barcelona - Sants,41.37880,2.1331,44.652174
229,2020-01-31,2020,1,31,4,Barcelona - Poblenou,41.40390,2.2045,58.478261
230,2020-01-31,2020,1,31,57,Barcelona - Palau Reial,41.38750,2.1151,40.043478


In [19]:
def process_all_pollution_recursive(data_folder: str | Path) -> pd.DataFrame:
    """Concatenate all csvs into one data frame with clean data.

    Args:
        data_folder (str | Path): Path to the folder with the data.

    Returns:
        pd.DataFrame: Data frame with al files' cleaned data.

    """
    files = sorted(Path(data_folder).rglob('*.csv'))
    results = []

    for file in files:
        df_single = process_pollution_file(file)
        df_clean = clean_final_columns(df_single)
        results.append(df_clean)

    return pd.concat(results, ignore_index=True)

In [20]:
total_data_set = process_all_pollution_recursive('datasets/air_quality/2020s/')
total_data_set

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2019-04-02,2019,4,2,4,Barcelona - Poblenou,41.40390,2.2045,53.954545
1,2019-04-02,2019,4,2,57,Barcelona - Palau Reial,41.38750,2.1151,35.125000
2,2019-04-02,2019,4,2,42,Barcelona - Sants,41.37880,2.1331,38.708333
3,2019-04-02,2019,4,2,54,Barcelona - Vall Hebron,41.42610,2.1480,50.333333
4,2019-04-02,2019,4,2,43,Barcelona - Eixample,41.38530,2.1538,52.875000
...,...,...,...,...,...,...,...,...,...
16417,2024-12-31,2024,12,31,43,Barcelona - Eixample,41.38530,2.1538,51.708333
16418,2024-12-31,2024,12,31,42,Barcelona - Sants,41.37880,2.1331,38.666667
16419,2024-12-31,2024,12,31,4,Barcelona - Poblenou,41.40390,2.2045,45.666667
16420,2024-12-31,2024,12,31,57,Barcelona - Palau Reial,41.38750,2.1151,30.583333


In [21]:
# Grab test file for pre-2020 data strucutre
data_f_19 = pd.read_csv(TEST_2019)

In [22]:
def clean_no2_value(value: str) -> float | None:
    """Clean the NO2 value from pre-2020 CSV format.

    Args:
        value (str): The no2 value to alter.

    Returns:
        float | None: Numeric NO2 value or None if missing.

    """
    # Check if the values are valid.
    if pd.isna(value):
        return None

    # Remove spaces
    value = str(value).strip()

    if value == '--':
        return None

    # Remove units
    value = value.replace('µg/m³', '')
    value = value.strip()

    # Convert to float
    try:
        return float(value)
    except ValueError:
        return None

In [23]:
data_f_19['no2_value'] = data_f_19['valor_no2'].apply(clean_no2_value)
data_f_19['no2_value']

0        86.0
1       112.0
2       113.0
3        74.0
4        67.0
        ...  
5899      NaN
5900      NaN
5901      NaN
5902      NaN
5903      NaN
Name: no2_value, Length: 5904, dtype: float64

In [24]:
def parse_p2020_datetime(data_f: pd.DataFrame) -> pd.DataFrame:
    """Recreate a usable date column for the pre-202 data.

    Args:
        data_f (pd.DataFrame): Data frame to alter.

    Returns:
        pd.DataFrame: Data frame with usable datae added.

    """
    data_f = data_f.copy()

    # Convert 'generat' to a usable format
    data_f['datetime'] = pd.to_datetime(data_f['generat'], format='%d/%m/%Y %H:%M')

    # Extract the date
    data_f['date'] = data_f['datetime'].dt.date

    return data_f

In [25]:
data_f_19 = parse_p2020_datetime(data_f_19)
data_f_19[['date']]


,date
0,2019-01-01
1,2019-01-01
2,2019-01-01
3,2019-01-01
4,2019-01-01
...,...
5899,2019-01-31
5900,2019-01-31
5901,2019-01-31
5902,2019-01-31


In [26]:
def build_station_mapping(station_file: str = STATION_FILE) -> pd.DataFrame:
    """Map station codes to numbers for pre-2020 structure.

    Args:
        station_file (str, optional): Station csv. Defaults to STATION_FILE.

    Returns:
        dict: the dictionary map of IDs and numbers.

    """
    data_f = pd.read_csv(station_file, sep=',')

    # Only keep NO2 stations
    df_pollutant = data_f[data_f['Codi_Contaminant'] == POLLUTANT_CODE].copy()

    # Keep just the key values
    station_map = df_pollutant[['codi_dtes', 'Estacio']].copy()

    return station_map.rename(columns={'Estacio': 'estacio'})

In [27]:
station_code_map = build_station_mapping()
station_code_map

,codi_dtes,estacio
0,IL,50
4,IH,43
11,IJ,44
18,IZ,57
25,I2,4
29,ID,42
32,IN,54
39,IZ,58


In [28]:
def map_dtes_to_station(
    data_f_p2020: pd.DataFrame, mapping_df: pd.DataFrame
) -> pd.DataFrame:
    """Map the code to the station number.

    Args:
        data_f_p2020 (pd.DataFrame): Pre-2020 data.
        mapping_df (pd.DataFrame): Mapped station dataframe.

    Returns:
        pd.DataFrame: Dataframe with stations added.

    """
    # First merge on codi_dtes where mapping is unique
    df_merged = data_f_p2020.merge(
        mapping_df.drop_duplicates('codi_dtes'), on='codi_dtes', how='left'
    )

    # Now fix the ambiguous IZ codes
    df_merged.loc[
        (df_merged['codi_dtes'] == 'IZ')
        & (df_merged['nom_cabina'].str.contains('Palau Reial')),
        'estacio',
    ] = 57

    df_merged.loc[
        (df_merged['codi_dtes'] == 'IZ')
        & (df_merged['nom_cabina'].str.contains('Fabra')),
        'estacio',
    ] = 58

    # Old files sometimes use OF for this
    df_merged.loc[
        (df_merged['codi_dtes'] == 'OF')
        & (df_merged['nom_cabina'].str.contains('Fabra')),
        'estacio',
    ] = 58

    return df_merged


In [29]:
mapped_p2020 = map_dtes_to_station(data_f_19, station_code_map)
mapped_p2020

,nom_cabina,qualitat_aire,codi_dtes,zqa,codi_eoi,longitud,latitud,hora_o3,qualitat_o3,valor_o3,...,valor_no2,hora_pm10,qualitat_pm10,valor_pm10,generat,dateTime,no2_value,datetime,date,estacio
0,Barcelona - Sants,Bona,ID,1,8019042,2.1331,41.3788,NaN,NaN,NaN,...,86 µg/m³,NaN,NaN,NaN,01/01/2019 0:00,1546297502,86.0,2019-01-01 00:00:00,2019-01-01,42.0
1,Barcelona - Eixample,Regular,IH,1,8019043,2.1538,41.3853,23h,Bona,3 µg/m³,...,112 µg/m³,23h,Bona,36 µg/m³,01/01/2019 0:00,1546297502,112.0,2019-01-01 00:00:00,2019-01-01,43.0
2,Barcelona - Gràcia,Regular,IJ,1,8019044,2.1534,41.3987,23h,Bona,3 µg/m³,...,113 µg/m³,23h,Regular,39 µg/m³,01/01/2019 0:00,1546297502,113.0,2019-01-01 00:00:00,2019-01-01,44.0
3,Barcelona - Ciutadella,Bona,IL,1,8019050,2.1874,41.3864,23h,Bona,2 µg/m³,...,74 µg/m³,NaN,NaN,NaN,01/01/2019 0:00,1546297502,74.0,2019-01-01 00:00:00,2019-01-01,50.0
4,Barcelona - Vall Hebron,Bona,IN,1,8019054,2.1480,41.4261,23h,Bona,10 µg/m³,...,67 µg/m³,23h,Bona,12 µg/m³,01/01/2019 0:00,1546297502,67.0,2019-01-01 00:00:00,2019-01-01,54.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5899,Barcelona - Ciutadella,--,IL,1,8019050,2.1874,41.3864,20h,--,--,...,--,NaN,NaN,NaN,31/01/2019 23:00,1548972301,NaN,2019-01-31 23:00:00,2019-01-31,50.0
5900,Barcelona - Vall Hebron,--,IN,1,8019054,2.1480,41.4261,20h,--,--,...,--,20h,--,--,31/01/2019 23:00,1548972301,NaN,2019-01-31 23:00:00,2019-01-31,54.0
5901,Barcelona - Palau Reial,--,IZ,1,8019057,2.1151,41.3875,20h,--,--,...,--,20h,--,--,31/01/2019 23:00,1548972301,NaN,2019-01-31 23:00:00,2019-01-31,57.0
5902,Barcelona - Poblenou,--,I2,1,8019004,2.2045,41.4039,NaN,NaN,NaN,...,--,20h,--,--,31/01/2019 23:00,1548972301,NaN,2019-01-31 23:00:00,2019-01-31,4.0


In [30]:
def calculate_p2020_daily_avg(clean_date_f: pd.DataFrame) -> pd.DataFrame:
    """Calculate the pre-2020 average NO2.

    Args:
        clean_date_f (pd.DataFrame): Dataframe to use fo calculating averages.

    Returns:
        pd.DataFrame: Data frame with NO2 averages.

    """
    clean_date_f = clean_date_f.copy()
    clean_date_f['date'] = pd.to_datetime(clean_date_f['date'])
    # DOn't use rows with missing values.
    clean_date_f = clean_date_f[clean_date_f['no2_value'].notna()].copy()

    return (
        clean_date_f.groupby(
            ['nom_cabina', 'latitud', 'longitud', 'date', 'estacio']
        ).agg(no2_daily_avg=('no2_value', 'mean'))
    ).reset_index()


In [31]:
avg_p2020_df = calculate_p2020_daily_avg(mapped_p2020)
avg_p2020_df

,nom_cabina,latitud,longitud,date,estacio,no2_daily_avg
0,Barcelona - Ciutadella,41.3864,2.1874,2019-01-01,50.0,38.708333
1,Barcelona - Ciutadella,41.3864,2.1874,2019-01-02,50.0,41.391304
2,Barcelona - Ciutadella,41.3864,2.1874,2019-01-03,50.0,47.285714
3,Barcelona - Ciutadella,41.3864,2.1874,2019-01-04,50.0,44.130435
4,Barcelona - Ciutadella,41.3864,2.1874,2019-01-05,50.0,47.809524
...,...,...,...,...,...,...
243,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-27,54.0,25.708333
244,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-28,54.0,19.750000
245,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-29,54.0,27.916667
246,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-30,54.0,18.791667


In [32]:
def finalize_p2020_file(df_p2020: pd.DataFrame) -> pd.DataFrame:
    """Produce dataframe organised like 2020+ files.

    Args:
        df_p2020 (pd.DataFrame): Pre-2020 data files

    Returns:
        pd.DataFrame: Dataframe organised like 2020+ files.

    """
    data_f = df_p2020.copy()
    data_f['date'] = pd.to_datetime(data_f['date'])
    data_f['year'] = data_f['date'].dt.year
    data_f['month'] = data_f['date'].dt.month
    data_f['day'] = data_f['date'].dt.day

    data_f['estacio'] = data_f['estacio'].astype('Int64')

    # Rename to match 2020+ dataset
    df_format = data_f.rename(
        columns={'nom_cabina': 'station_name', 'latitud': 'lat', 'longitud': 'lon'}
    )

    cols = [
        'date',
        'year',
        'month',
        'day',
        'estacio',
        'station_name',
        'lat',
        'lon',
        'no2_daily_avg',
    ]

    # Structure to follow same as 2020+ files.
    df_format = df_format[cols].sort_values(['date', 'estacio'])

    return df_format[cols].sort_values(['date', 'estacio']).reset_index(drop=True)


In [33]:
final_p2020_frame = finalize_p2020_file(avg_p2020_df)
final_p2020_frame

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2019-01-01,2019,1,1,4,Barcelona - Poblenou,41.4039,2.2045,42.391304
1,2019-01-01,2019,1,1,42,Barcelona - Sants,41.3788,2.1331,54.833333
2,2019-01-01,2019,1,1,43,Barcelona - Eixample,41.3853,2.1538,73.650000
3,2019-01-01,2019,1,1,44,Barcelona - Gràcia,41.3987,2.1534,56.714286
4,2019-01-01,2019,1,1,50,Barcelona - Ciutadella,41.3864,2.1874,38.708333
...,...,...,...,...,...,...,...,...,...
243,2019-01-31,2019,1,31,44,Barcelona - Gràcia,41.3987,2.1534,40.000000
244,2019-01-31,2019,1,31,50,Barcelona - Ciutadella,41.3864,2.1874,43.062500
245,2019-01-31,2019,1,31,54,Barcelona - Vall Hebron,41.4261,2.1480,27.692308
246,2019-01-31,2019,1,31,57,Barcelona - Palau Reial,41.3875,2.1151,23.933333


In [34]:
def process_pre2020_file(
    file_path: str | Path, station_map: pd.DataFrame
) -> pd.DataFrame:
    """Process a pre-2020 file for combining with 2020+.

    Args:
        file_path (str | Path): path ro file to process
        station_map (pd.DataFrame): Dataframe with mapped codes

    Returns:
        pd.DataFrame: Final dataframe.

    """
    # Run through the functions created previously to create the relevant structure.
    data_f = pd.read_csv(file_path)
    data_f['no2_value'] = data_f['valor_no2'].apply(clean_no2_value)
    data_f = parse_p2020_datetime(data_f)
    data_f = map_dtes_to_station(data_f, station_map)
    daily_df = calculate_p2020_daily_avg(data_f)

    return finalize_p2020_file(daily_df)


In [35]:
process_pre2020_file(TEST_2019, station_code_map)

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2019-01-01,2019,1,1,4,Barcelona - Poblenou,41.4039,2.2045,42.391304
1,2019-01-01,2019,1,1,42,Barcelona - Sants,41.3788,2.1331,54.833333
2,2019-01-01,2019,1,1,43,Barcelona - Eixample,41.3853,2.1538,73.650000
3,2019-01-01,2019,1,1,44,Barcelona - Gràcia,41.3987,2.1534,56.714286
4,2019-01-01,2019,1,1,50,Barcelona - Ciutadella,41.3864,2.1874,38.708333
...,...,...,...,...,...,...,...,...,...
243,2019-01-31,2019,1,31,44,Barcelona - Gràcia,41.3987,2.1534,40.000000
244,2019-01-31,2019,1,31,50,Barcelona - Ciutadella,41.3864,2.1874,43.062500
245,2019-01-31,2019,1,31,54,Barcelona - Vall Hebron,41.4261,2.1480,27.692308
246,2019-01-31,2019,1,31,57,Barcelona - Palau Reial,41.3875,2.1151,23.933333


In [36]:
def process_pre2020_all(folder_path: str, station_map: pd.DataFrame) -> pd.DataFrame:
    """Loop through folder and generate necessary structure for merging.

    Args:
        folder_path (str): path to folder with files to process.
        station_map (pd.DataFrame): Map of station names and codes.

    Returns:
        pd.DataFrame: Formatted, merged output.

    """
    files = sorted(Path(folder_path).rglob('*.csv'))
    results = []

    for file in files:
        df = process_pre2020_file(file, station_map)
        results.append(df)

    return pd.concat(results, ignore_index=True)


In [37]:
p2020_total = process_pre2020_all('datasets/air_quality/pre_2020/', station_code_map)
p2020_total

/var/folders/x8/v0_l4_492z59nt9n8s65c3sw0000gp/T/ipykernel_5059/3485863898.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(results, ignore_index=True)


,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2018-06-11,2018,6,11,4,Barcelona - Poblenou,41.4039,2.2045,42.800000
1,2018-06-11,2018,6,11,42,Barcelona - Sants,41.3788,2.1331,16.357143
2,2018-06-11,2018,6,11,43,Barcelona - Eixample,41.3853,2.1538,59.600000
3,2018-06-11,2018,6,11,44,Barcelona - Gràcia,41.3987,2.1534,43.200000
4,2018-06-11,2018,6,11,50,Barcelona - Ciutadella,41.3864,2.1874,35.230769
...,...,...,...,...,...,...,...,...,...
1749,2019-02-01,2019,2,1,44,Barcelona - Gràcia,41.3987,2.1534,73.000000
1750,2019-02-01,2019,2,1,50,Barcelona - Ciutadella,41.3864,2.1874,33.000000
1751,2019-02-01,2019,2,1,54,Barcelona - Vall Hebron,41.4261,2.1480,34.000000
1752,2019-02-01,2019,2,1,57,Barcelona - Palau Reial,41.3875,2.1151,19.000000


In [38]:
combined_all_years = pd.concat([p2020_total, total_data_set], ignore_index=True)
combined_all_years


,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2018-06-11,2018,6,11,4,Barcelona - Poblenou,41.40390,2.2045,42.800000
1,2018-06-11,2018,6,11,42,Barcelona - Sants,41.37880,2.1331,16.357143
2,2018-06-11,2018,6,11,43,Barcelona - Eixample,41.38530,2.1538,59.600000
3,2018-06-11,2018,6,11,44,Barcelona - Gràcia,41.39870,2.1534,43.200000
4,2018-06-11,2018,6,11,50,Barcelona - Ciutadella,41.38640,2.1874,35.230769
...,...,...,...,...,...,...,...,...,...
18171,2024-12-31,2024,12,31,43,Barcelona - Eixample,41.38530,2.1538,51.708333
18172,2024-12-31,2024,12,31,42,Barcelona - Sants,41.37880,2.1331,38.666667
18173,2024-12-31,2024,12,31,4,Barcelona - Poblenou,41.40390,2.2045,45.666667
18174,2024-12-31,2024,12,31,57,Barcelona - Palau Reial,41.38750,2.1151,30.583333


In [41]:
combined_all_years["date"] = pd.to_datetime(combined_all_years["date"])
combined_all_years = combined_all_years.sort_values(["date", "estacio"]).reset_index(drop=True)
combined_all_years.to_csv('datasets/final/pollution_dataset.csv')
